# 07 · RAG & Embeddings

> **Source notes:** `RAGAndEmbeddings.md`

This notebook demonstrates:
- **Semantic search** via embeddings (all-MiniLM-L6-v2)
- **Retrieval-augmented generation (RAG)** — inject live context into LLM prompts
- **Chunk + rank + retrieve** pattern with zero external dependencies (no Pinecone, FAISS, or ChromaDB)

All tools are local (Sentence Transformers + Ollama), no API keys required. Running example: *DocBot* — answering questions about Python documentation chunks.

## 1 · What Are Embeddings?

**Embeddings** transform text into fixed-size numerical vectors where *meaning becomes measurable*: semantically similar text clusters together in high-dimensional space.

### How a Transformer Encoder Produces an Embedding

```
Input text
  --> Tokenise  (split into word-pieces, add [CLS] and [SEP])
  --> Token + Position Embeddings
  --> Self-Attention x N layers   (every token attends to every other token -- O(n^2))
  --> Final hidden states  [N_tokens x 768]
  --> Pooling  (collapse to a single vector)
  --> Embedding  [768]
```

Key contrast: **encoder** models (BERT, all-MiniLM) process the entire input at once.  
**Decoder** models (GPT, Llama, Phi) process left-to-right and are used for generation.

### Pooling Strategies

| Strategy | Mechanism | When Used |
|----------|-----------|-----------|
| **[CLS] token** | Final hidden state of the special first token | BERT-family models |
| **Mean pooling** | Average all token embeddings | Most modern embedding models |
| **Max pooling** | Element-wise max across tokens | Specialised retrieval |
| **Last-token** | Final non-padding token hidden state | Decoder-based embeddings |

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

docs = [
    "A classic Margherita with fresh mozzarella and basil",
    "Pepperoni pizza with extra cheese",
    "Vegetarian pizza with mushrooms, peppers, and olives",
    "Hawaiian pizza with pineapple and ham",
    "BBQ chicken pizza with red onions"
]

query = "something light and vegetarian"

# Embed
doc_embeddings = model.encode(docs)
query_embedding = model.encode(query)

# Cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
scores = cosine_similarity([query_embedding], doc_embeddings)[0]

# Sort and print
results = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

print(f"Query: '{query}'\n")
for doc, score in results:
    print(f"{score:.3f} — {doc}")

## 2 · Contrastive Learning — How Embedding Models Are Trained

Embedding models are **not** trained to predict the next token.  
They use **contrastive learning**: push similar texts together, push unrelated texts apart.

The most common objective is **InfoNCE loss**:

$$
\mathcal{L} = -\log \frac{\exp(\text{sim}(q, p^+) / \tau)}{\exp(\text{sim}(q, p^+) / \tau) + \sum_i \exp(\text{sim}(q, p_i^-) / \tau)}
$$

- $q$ = query embedding, $p^+$ = positive (similar), $p_i^-$ = negatives
- $\tau$ = temperature (controls how "peaked" the distribution is)

In practice, positive pairs come from: (question, answer), (sentence, paraphrase), (chunk, its document title).

## 3 · The RAG Pipeline

RAG (Retrieval-Augmented Generation) gives the LLM access to a private or up-to-date corpus:

```
INGESTION (offline)                     QUERY (runtime)
-------------------                     ---------------
Documents                               User question
  --> chunk                               --> embed question
  --> embed each chunk                    --> ANN search --> top-k chunks
  --> store in vector DB                  --> build prompt (context + question)
                                          --> LLM generates grounded answer
```

Two phases:
1. **Ingestion** — split documents into chunks, embed, store
2. **Query** — embed the question, retrieve similar chunks, augment the LLM prompt

In [ ]:
import ollama
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

corpus = [
    "Margherita: tomato sauce, mozzarella, basil. Calories: 680. Price: $15.99",
    "Pepperoni: tomato sauce, mozzarella, pepperoni. Calories: 890. Price: $17.99",
    "Veggie: tomato sauce, mozzarella, mushrooms, peppers, olives. Calories: 620. Price: $16.99"
]

question = "What is the lowest calorie pizza and how much does it cost?"

# No context
response_no_context = ollama.generate(
    model='phi3:mini',
    prompt=question,
    options={'temperature': 0.0}
)

# With RAG retrieval
corpus_embeddings = model.encode(corpus)
query_embedding = model.encode(question)

from sklearn.metrics.pairwise import cosine_similarity
scores = cosine_similarity([query_embedding], corpus_embeddings)[0]
top3_idx = np.argsort(scores)[-3:][::-1]
top3_chunks = [corpus[i] for i in top3_idx]

context = "\n".join(top3_chunks)
prompt_with_context = f"""Context:
{context}

Question: {question}
Answer based only on the context above:"""

response_with_context = ollama.generate(
    model='phi3:mini',
    prompt=prompt_with_context,
    options={'temperature': 0.0}
)

print("WITHOUT RAG (no context):")
print(response_no_context['response'][:200])
print("\n" + "="*60 + "\n")
print("WITH RAG (grounded in context):")
print(response_with_context['response'][:200])

## 4 · Advanced RAG — HyDE (Hypothetical Document Embeddings)

**Problem:** A user question (*"What is the average rail speed?"*) and its answer document (*"The average speed is 57.5 km/h..."*) are phrased differently. This **semantic gap** degrades retrieval.

**HyDE solution:**
1. Ask the LLM to generate a *hypothetical answer document* (even if some details are approximate)
2. Embed **that hypothetical document** instead of the raw question
3. Search with the hypothetical embedding — it matches real documents much better

```
User query  --(LLM)-->  Hypothetical doc  --(embed)-->  vector  --(search)-->  real docs
                         (plausible answer)
```

This works because the generated answer has the same **phrasing style** as the real answer documents.

### Other Advanced Patterns

| Pattern | What It Does |
|---------|-------------|
| **FLARE** | Re-retrieves mid-generation when confidence drops |
| **Query Decomposition** | Breaks complex queries into sub-queries, retrieves for each |
| **Re-ranking** | Cross-encoder re-scores retrieved chunks for precision |
| **Sparse + Dense (Hybrid)** | BM25 keyword search combined with dense vector search |

In [ ]:
import ollama
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

corpus = [
    "The Margherita pizza features fresh mozzarella, tomato sauce, and basil leaves.",
    "Our pepperoni classic comes loaded with spicy pepperoni slices.",
    "Try our vegetarian option with mushrooms, bell peppers, and black olives.",
    "Hawaiian pizza includes pineapple chunks and Canadian bacon.",
    "BBQ chicken pizza has grilled chicken, red onions, and tangy BBQ sauce."
]

query = "What's a good light vegetarian option?"

corpus_embeddings = model.encode(corpus)

# Method 1: Raw query
query_embedding_raw = model.encode(query)
scores_raw = cosine_similarity([query_embedding_raw], corpus_embeddings)[0]
top3_raw = np.argsort(scores_raw)[-3:][::-1]

# Method 2: HyDE (generate hypothetical answer, embed that)
hyde_prompt = f"Answer this question as if you're reading from a menu: {query}"
hyde_doc = ollama.generate(
    model='phi3:mini',
    prompt=hyde_prompt,
    options={'temperature': 0.7, 'num_predict': 50}
)['response']

query_embedding_hyde = model.encode(hyde_doc)
scores_hyde = cosine_similarity([query_embedding_hyde], corpus_embeddings)[0]
top3_hyde = np.argsort(scores_hyde)[-3:][::-1]

print(f"Query: '{query}'")
print(f"\nHyDE generated doc: '{hyde_doc[:100]}...'\n")
print("="*60)
print("\nRaw query retrieval:")
for i in top3_raw:
    print(f"  {scores_raw[i]:.3f} — {corpus[i]}")

print("\n" + "="*60 + "\n")
print("HyDE retrieval:")
for i in top3_hyde:
    print(f"  {scores_hyde[i]:.3f} — {corpus[i]}")

## 5 · Production Failure Modes

| Failure Mode | Symptom | Fix |
|-------------|---------|-----|
| **Semantic gap** | Right answer exists but wrong chunks retrieved | Use HyDE or query expansion |
| **Chunk too large** | Relevant sentence diluted in big chunk | Smaller chunks; hierarchical chunking |
| **Lost-in-the-middle** | LLM ignores chunks in the middle of context | Put most relevant chunk first AND last |
| **Context overflow** | Too many chunks; noise drowns signal | Reduce k; use a cross-encoder re-ranker |
| **No answer in corpus** | LLM hallucinates when corpus is silent | Add explicit fallback instruction |
| **Unfaithful generation** | LLM answers from memory not context | Strong grounding instruction in prompt |

## 6 · Key Takeaways

| Concept | One-Liner |
|---------|-----------|
| Embedding | Text → fixed vector; similar meaning → close in space |
| Pooling | Mean pooling outperforms [CLS] on most tasks |
| Contrastive learning | Push similar together, push unrelated apart |
| RAG pipeline | Chunk then embed offline; embed + retrieve at query time |
| HyDE | Embed a hypothetical answer, not the raw question |

**Next:** `03-vector-dbs/notebook.ipynb` — how the vector store finds those chunks in milliseconds.

In [ ]:
import ollama
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

chunks = [
    "Chunk 1: General pizza info",
    "Chunk 2: More general info",
    "ALLERGEN WARNING: Margherita contains dairy",  # Target chunk
    "Chunk 4: General info continued",
    "Chunk 5: Closing info"
]

question = "Is Margherita safe for dairy allergies?"

positions_to_test = [0, 2, 4]  # position 0=first, 2=middle, 4=last

for pos in positions_to_test:
    # Reorder chunks to place target at position
    reordered = chunks.copy()
    target = "ALLERGEN WARNING: Margherita contains dairy"
    reordered.remove(target)
    reordered.insert(pos, target)

    context = "\n".join(reordered)
    prompt = f"""Context:
{context}

Question: {question}
Answer:"""

    response = ollama.generate(
        model='phi3:mini',
        prompt=prompt,
        options={'temperature': 0.0}
    )

    caught = "dairy" in response['response'].lower() or "not safe" in response['response'].lower()
    print(f"Target at position {pos}: {'✓ CAUGHT' if caught else '✗ MISSED'}")
    print(f"  Response: {response['response'][:100]}...\n")

## 6 · PizzaBot Connection — The RAG Corpus in Practice

> Full system spec: [AIPrimer.md](../AIPrimer.md)

The PizzaBot RAG corpus is a concrete, small-scale instance of every decision in this notebook:

| Decision | PizzaBot choice | Notebook section |
|---|---|---|
| **Embedding model** | `all-MiniLM-L6-v2` (384d) — same model at ingestion and query time | §1 |
| **Chunking strategy** | Per-item for `menu.json` and `allergens.csv`; per-section for `faq.md` | §RAGAndEmbeddings.md §8 |
| **Same-model constraint** | The query `"something light and vegetarian"` uses the same MiniLM weights that embedded the Veggie Feast entry | §2 |
| **Semantic retrieval** | Query `"light vegetarian"` retrieves Veggie Feast even though those exact words don't appear in `menu.json` | §3 |
| **HyDE** | Generates a hypothetical answer (`"A light vegetarian option would be a Margherita or a garden salad pizza..."`) and embeds *that* to retrieve the closest menu item | §4 |
| **Failure mode: lost-in-middle** | Long conversation with 5 retrieved chunks: the allergen chunk in position 3 (middle) is most likely to be ignored by the model | §5 |

**Try it:** replace the ChromaDB documents in §3 with the PizzaBot corpus files from `projects/ai/rag-pipeline/` when that project is completed.
